# Final DBSCAN + logistic refit on 235 verified trials

User-authorized final training for a manual Kaggle submission. Hyperparameters are frozen from the earlier experiments. This notebook never submits to Kaggle.

**Post-submission update:** Michael reported Kaggle **0.86452**. The unchanged CSV is archived under `submissions/archive/submitted/dbscan_logistic_trainval235__kaggle_0.86452.csv`. The earlier 0.910256 validation score belongs to the 157-label model.

## Frozen settings and final fitting population

Combine the 157 verified training labels with all 78 former validation labels. Keep the 1,014 originally unlabeled training trials. Test trials are excluded from all fitting. Upload is manual.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import DBSCAN
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

REASONS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
FEATURES = [f'emb_{i}' for i in range(150)]
cwd = Path.cwd()
DATA = next(p for p in [cwd, cwd/'part1', cwd/'AML-Midterm'/'part1']
            if (p/'part1_train.csv').is_file() or (p/'data/part1_train.csv').is_file()).resolve()
INPUT = DATA if (DATA/'part1_train.csv').is_file() else DATA/'data'
OUT = DATA/'experiments/final_dbscan_trainval235'
READY = DATA/'submissions/ready'
for p in [OUT, OUT/'datasets', OUT/'models', OUT/'predictions', READY]:
    p.mkdir(parents=True, exist_ok=True)
source = DATA/'experiments/additional_methods'
source_manifest = json.loads((source/'submission_manifest.json').read_text())
source_classifier = source/source_manifest['model_file']
source_cluster = source/'labeling_models/dbscan_m5_q25.joblib'
old_model = joblib.load(source_classifier)
old_cluster = joblib.load(source_cluster)
LABEL_RULE = old_cluster['label_rule'].copy()
assert LABEL_RULE == dict(minimum_support=3., prior_strength=2., positive_threshold=0.6,
                          negative_threshold=0.1, maximum_generated_weight=0.25,
                          maximum_class_fraction=0.5)
assert old_cluster['model'].min_samples == 5 and old_cluster['eps_quantile'] == 0.25
for estimator in old_model['estimators']:
    assert estimator.C == 1000. and estimator.tol == 1e-4
    assert estimator.solver == 'lbfgs' and estimator.max_iter == 5000
    assert estimator.class_weight is None and estimator.l1_ratio == 0.
RECIPE = dict(features=FEATURES, C=1000., tol=1e-4, solver='lbfgs', max_iter=5000,
              random_state=42, class_weight=None, penalty='L2',
              dbscan_min_samples=5, dbscan_radius_quantile=0.25, dbscan_radius_floor=1e-6,
              label_rule=LABEL_RULE, threshold=0.5, highest_probability_fallback=True,
              row_normalization=False, additional_pca=False)
train = pd.read_csv(INPUT/'part1_train.csv')
validation = pd.read_csv(INPUT/'part1_val.csv')
test = pd.read_csv(INPUT/'part1_test.csv')
sample = pd.read_csv(INPUT/'part1_sample_submission.csv')
assert len(train) == 1171 and len(validation) == 78 and len(test) == len(sample) == 312
assert train.ID.is_unique and validation.ID.is_unique and test.ID.is_unique
assert set(train.ID).isdisjoint(validation.ID)
assert set(test.ID).isdisjoint(set(train.ID) | set(validation.ID))
assert test.ID.tolist() == sample.ID.tolist() and sample.columns.tolist() == ['ID', 'Predicted']
assert train[REASONS].isin([0, 1]).all(axis=1).sum() == 157
assert validation[REASONS].isin([0, 1]).all(axis=1).all()
combined = pd.concat([train, validation], ignore_index=True)
X = combined[FEATURES].to_numpy()
Y = combined[REASONS].to_numpy(dtype=int)
known = np.isin(Y, [0, 1]).all(axis=1)
XT = test[FEATURES].to_numpy()
assert len(combined) == 1249 and known.sum() == 235 and (~known).sum() == 1014
assert (Y[~known] == -100).all() and np.isfinite(X).all() and np.isfinite(XT).all()
protected_paths = [INPUT/f for f in ['part1_train.csv', 'part1_val.csv', 'part1_test.csv', 'part1_sample_submission.csv']]
protected_paths += [source_classifier, source_cluster, source/'submission_manifest.json',
                    source/'datasets/dbscan_m5_q25.csv', source/source_manifest['submission_file']]
PROTECTED = {str(p.relative_to(DATA)): hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
combined[['ID']+FEATURES+REASONS].to_csv(OUT/'datasets/train_plus_validation.csv', index=False)
test[['ID']+FEATURES].to_csv(OUT/'datasets/test_features.csv', index=False)
print('Frozen recipe: DBSCAN min_samples=5 / q25 radius; five L2 logistic classifiers, C=1000, tol=1e-4.')
print('Final fitting data: 235 verified labels (157 training + 78 validation), plus 1,014 unlabeled trials.')
print('Kaggle upload is manual. This notebook makes no external submission.')



Frozen recipe: DBSCAN min_samples=5 / q25 radius; five L2 logistic classifiers, C=1000, tol=1e-4.
Final fitting data: 235 verified labels (157 training + 78 validation), plus 1,014 unlabeled trials.
Kaggle upload is manual. This notebook makes no external submission.


## Labeling and prediction helpers; historical reference

Reproduce the untouched selection model’s earlier validation scores for provenance. They do not measure the final refit.

In [2]:
def predict_model(packet, raw):
    scaled = packet['scaler'].transform(np.asarray(raw))
    return np.column_stack([est.predict_proba(scaled)[:, list(est.classes_).index(1)]
                            for est in packet['estimators']])


def label_predictions(probability):
    assert probability.ndim == 2 and probability.shape[1] == 5
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    prediction = (probability >= 0.5).astype(int)
    empty = prediction.sum(axis=1) == 0
    rows = np.flatnonzero(empty)
    prediction[rows, probability[rows].argmax(axis=1)] = 1
    return prediction, empty


def generate_labels(labels, verified, probability, support):
    targets = np.full(labels.shape, -100, dtype=int)
    weights = np.zeros(labels.shape)
    targets[verified] = labels[verified]
    weights[verified] = 1.
    eligible = (~verified) & (support >= LABEL_RULE['minimum_support'])
    for j in range(5):
        positive = eligible & (probability[:, j] >= LABEL_RULE['positive_threshold'])
        negative = eligible & (probability[:, j] <= LABEL_RULE['negative_threshold'])
        npos, nneg = int(positive.sum()), int(negative.sum())
        kp = int(labels[verified, j].sum())
        kn = int(verified.sum() - kp)
        assert kp > 0 and kn > 0
        if not npos or not nneg:
            continue
        fraction = min(LABEL_RULE['maximum_class_fraction'],
                       LABEL_RULE['maximum_generated_weight']*npos/kp,
                       LABEL_RULE['maximum_generated_weight']*nneg/kn)
        targets[positive, j] = 1
        targets[negative, j] = 0
        weights[positive, j] = fraction*kp/npos
        weights[negative, j] = fraction*kn/nneg
        assert weights[~verified, j].max() <= LABEL_RULE['maximum_generated_weight']+1e-12
        np.testing.assert_allclose(weights[positive, j].sum()/kp, weights[negative, j].sum()/kn, atol=1e-12)
    np.testing.assert_array_equal(targets[verified], labels[verified])
    np.testing.assert_array_equal(weights[verified], np.ones((verified.sum(), 5)))
    assert np.array_equal(weights > 0, targets != -100)
    return targets, weights


# Reconfirm provenance using the untouched 157-label model, not the final fit.
historical_probability = predict_model(old_model, validation[FEATURES].to_numpy())
historical_prediction, _ = label_predictions(historical_probability)
historical_sample = float(f1_score(validation[REASONS], historical_prediction, average='samples', zero_division=0))
historical_macro = float(f1_score(validation[REASONS], historical_prediction, average='macro', zero_division=0))
np.testing.assert_allclose([historical_sample, historical_macro],
                          [source_manifest['validation_sample_f1'], source_manifest['validation_macro_f1']], atol=1e-14)
print(f'Historical pre-refit validation result reproduced: sample F1={historical_sample:.6f}, macro F1={historical_macro:.6f}.')
print('Those are selection-model scores; the final model has no remaining local holdout.')



Historical pre-refit validation result reproduced: sample F1=0.910256, macro F1=0.811421.
Those are selection-model scores; the final model has no remaining local holdout.


## Refit preprocessing and regenerate labels

Fit classifier scaling on the 235 known trials and clustering preprocessing on all 1,249 train/validation trials. Relearn DBSCAN’s training-derived radius with the unchanged q25 rule and regenerate labels/weights.

In [3]:
classifier_scaler = StandardScaler().fit(X[known])
cluster_scaler = StandardScaler().fit(X)
XS = classifier_scaler.transform(X)
XC = cluster_scaler.transform(X)
assert int(classifier_scaler.n_samples_seen_) == 235 and int(cluster_scaler.n_samples_seen_) == 1249
with threadpool_limits(limits=1):
    distances = NearestNeighbors(n_neighbors=5, algorithm='brute', n_jobs=1).fit(XC).kneighbors(XC)[0][:, -1]
    eps = max(float(np.quantile(distances, RECIPE['dbscan_radius_quantile'])), RECIPE['dbscan_radius_floor'])
    clustering = DBSCAN(eps=eps, min_samples=5, metric='euclidean', algorithm='brute', n_jobs=1).fit(XC)
cluster_ids = clustering.labels_
clusters = np.unique(cluster_ids[cluster_ids >= 0])
responsibility = np.column_stack([cluster_ids == k for k in clusters]).astype(float) if len(clusters) else np.empty((len(X), 0))
component_support = responsibility[known].sum(axis=0)
component_positive = responsibility[known].T @ Y[known]
prior = Y[known].mean(axis=0)
rates = (component_positive + LABEL_RULE['prior_strength']*prior) / (component_support[:, None]+LABEL_RULE['prior_strength'])
probability = responsibility @ rates
support = responsibility @ component_support
targets, raw_weights = generate_labels(Y, known, probability, support)
assert (raw_weights[(cluster_ids < 0) & ~known] == 0).all()
normalized_weights = raw_weights*known.sum()/raw_weights.sum(axis=0)
np.testing.assert_allclose(normalized_weights.sum(axis=0), np.full(5, 235), atol=1e-10)
dataset = combined[['ID']+FEATURES].copy()
dataset['verified_label_source'] = np.where(np.arange(len(combined)) >= len(train), 'validation',
                                           np.where(known, 'training', 'unlabeled'))
audit = combined[['ID']].copy()
audit['dbscan_cluster'] = cluster_ids
audit['verified_label_source'] = dataset.verified_label_source
audit['effective_verified_support'] = support
label_counts = []
for j, reason in enumerate(REASONS):
    dataset[reason] = targets[:, j]
    dataset[f'weight_{reason}'] = raw_weights[:, j]
    dataset[f'normalized_weight_{reason}'] = normalized_weights[:, j]
    audit[f'original_{reason}'] = Y[:, j]
    audit[f'target_{reason}'] = targets[:, j]
    audit[f'estimated_probability_{reason}'] = np.where(cluster_ids >= 0, probability[:, j], np.nan)
    audit[f'label_source_{reason}'] = np.where(known, 'verified', np.where(raw_weights[:, j] > 0, 'generated', 'unknown'))
    positive = (~known) & (targets[:, j] == 1)
    negative = (~known) & (targets[:, j] == 0)
    label_counts.append(dict(reason=reason, verified_positive=int(Y[known, j].sum()),
                             verified_negative=int((Y[known, j] == 0).sum()),
                             generated_positive=int(positive.sum()), generated_negative=int(negative.sum()),
                             generated_positive_raw_weight=float(raw_weights[positive, j].sum()),
                             generated_negative_raw_weight=float(raw_weights[negative, j].sum()),
                             total_fitting_rows=int((raw_weights[:, j] > 0).sum()),
                             normalized_weight_total=float(normalized_weights[:, j].sum())))
dataset.to_csv(OUT/'datasets/dbscan_trainval235_targets_weights.csv', index=False)
audit.to_csv(OUT/'datasets/label_audit.csv', index=False)
counts = pd.DataFrame(label_counts)
counts.to_csv(OUT/'label_counts.csv', index=False)
cluster_audit = pd.DataFrame(dict(cluster=clusters, trials=responsibility.sum(axis=0), verified_support=component_support))
for j, reason in enumerate(REASONS):
    cluster_audit[f'probability_{reason}'] = rates[:, j]
cluster_audit.to_csv(OUT/'cluster_audit.csv', index=False)
joblib.dump(dict(model=clustering, scaler=cluster_scaler, eps_quantile=0.25,
                 label_rule=LABEL_RULE, fitting_IDs=combined.ID.tolist()), OUT/'models/dbscan_labeler_trainval235.joblib', compress=3)
print(f'Refitted DBSCAN on 1,249 training/validation rows: eps={eps:.15g}, clusters={len(clusters)}, noise={int((cluster_ids < 0).sum())}.')
display(counts)



Refitted DBSCAN on 1,249 training/validation rows: eps=0.00014595414084762, clusters=18, noise=935.


,reason,verified_positive,verified_negative,generated_positive,generated_negative,generated_positive_raw_weight,generated_negative_raw_weight,total_fitting_rows,normalized_weight_total
0,Safety,25,210,0,0,0.0,0.00000,235,235.0
1,Funding,12,223,0,0,0.0,0.00000,235,235.0
2,Efficacy,31,204,8,165,2.0,13.16129,408,235.0
3,Recruitment,151,84,0,0,0.0,0.00000,235,235.0
4,Other,35,200,0,0,0.0,0.00000,235,235.0


## Train the final five reason classifiers

Retain the original L2 logistic settings, including stopping tolerance. Normalize each reason’s fitting weights to total 235.

In [4]:
estimators = []
fit_audit = []
with threadpool_limits(limits=1):
    for j, reason in enumerate(REASONS):
        included = raw_weights[:, j] > 0
        estimator = clone(old_model['estimators'][j])
        with warnings.catch_warnings():
            warnings.simplefilter('error', ConvergenceWarning)
            estimator.fit(XS[included], targets[included, j], sample_weight=normalized_weights[included, j])
        estimators.append(estimator)
        fit_audit.append(dict(reason=reason, verified_rows=235, fitting_rows=int(included.sum()),
                              generated_rows=int((included & ~known).sum()),
                              weight_total=float(normalized_weights[included, j].sum()),
                              iterations=int(estimator.n_iter_.max()), C=float(estimator.C),
                              tol=float(estimator.tol), solver=estimator.solver))
model = dict(kind='logistic', parameter=1000., scaler=classifier_scaler, estimators=estimators,
             features=FEATURES, reasons=REASONS, dataset='dbscan_trainval235',
             logistic_weight_normalization_total=235, verified_rows=235,
             train_plus_validation_refit=True, prediction_cutoff=0.5, nonempty_fallback=True,
             recipe=RECIPE)
model_path = OUT/'models/dbscan_logistic_trainval235.joblib'
joblib.dump(model, model_path, compress=3)
pd.DataFrame(fit_audit).to_csv(OUT/'fit_audit.csv', index=False)
print('All five classifiers fitted with the original solver, tolerance, and regularization settings.')
display(pd.DataFrame(fit_audit))



All five classifiers fitted with the original solver, tolerance, and regularization settings.


,reason,verified_rows,fitting_rows,generated_rows,weight_total,iterations,C,tol,solver
0,Safety,235,235,0,235.0,15,1000.0,0.0001,lbfgs
1,Funding,235,235,0,235.0,13,1000.0,0.0001,lbfgs
2,Efficacy,235,408,173,235.0,15,1000.0,0.0001,lbfgs
3,Recruitment,235,235,0,235.0,14,1000.0,0.0001,lbfgs
4,Other,235,235,0,235.0,16,1000.0,0.0001,lbfgs


## Generate the manual-upload CSV

Predict the 312 test trials, enforce the existing cutoff and nonempty fallback, and verify exact submission IDs/schema.

In [5]:
test_probability = predict_model(model, XT)
test_prediction, empty = label_predictions(test_probability)
submission = pd.DataFrame(dict(ID=test.ID, Predicted=[
    ' '.join(reason for reason, keep in zip(REASONS, row) if keep) for row in test_prediction]))
submission_path = DATA/'submissions/archive/submitted/dbscan_logistic_trainval235__kaggle_0.86452.csv'
submission.to_csv(submission_path, index=False)
saved_submission = pd.read_csv(submission_path)
pd.testing.assert_frame_equal(saved_submission, submission)
assert saved_submission.columns.tolist() == sample.columns.tolist() == ['ID', 'Predicted']
assert saved_submission.ID.tolist() == sample.ID.tolist() and saved_submission.ID.is_unique
assert saved_submission.Predicted.notna().all() and saved_submission.Predicted.str.strip().ne('').all()
assert all(set(s.split()).issubset(REASONS) and len(s.split()) == len(set(s.split())) for s in saved_submission.Predicted)
np.testing.assert_array_equal(np.array([[int(r in s.split()) for r in REASONS]
                                      for s in saved_submission.Predicted]), test_prediction)
prediction_frame = test[['ID']].copy()
for j, reason in enumerate(REASONS):
    prediction_frame[f'probability_{reason}'] = test_probability[:, j]
    prediction_frame[f'predicted_{reason}'] = test_prediction[:, j]
prediction_frame['used_fallback'] = empty
prediction_frame.to_csv(OUT/'predictions/test_probabilities.csv', index=False)
old_test_prediction = label_predictions(predict_model(old_model, XT))[0]
changed_test_trials = int((test_prediction != old_test_prediction).any(axis=1).sum())
print(f'Saved {len(submission)} predictions; fallback used on {int(empty.sum())} trials.')
print(f'{changed_test_trials} test trials changed relative to the earlier 157-label DBSCAN model.')
print('Saved final-fit submission CSV:', submission_path)



Saved 312 predictions; fallback used on 17 trials.
34 test trials changed relative to the earlier 157-label DBSCAN model.
Ready for manual upload: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/submissions/ready/part1_dbscan_logistic_trainval235.csv


## Reload and independently refit

Verify saved targets/weights, model prediction replay, an independent classifier refit, and protected source checksums.

In [6]:
reloaded = joblib.load(model_path)
np.testing.assert_allclose(predict_model(reloaded, XT), test_probability, rtol=1e-12, atol=1e-12)
reloaded_labeler = joblib.load(OUT/'models/dbscan_labeler_trainval235.joblib')
np.testing.assert_array_equal(reloaded_labeler['model'].labels_, cluster_ids)
saved = pd.read_csv(OUT/'datasets/dbscan_trainval235_targets_weights.csv', float_precision='round_trip')
assert saved.ID.tolist() == combined.ID.tolist() and set(saved.ID).isdisjoint(test.ID)
np.testing.assert_array_equal(saved[REASONS].to_numpy(dtype=int), targets)
np.testing.assert_allclose(saved[[f'weight_{r}' for r in REASONS]], raw_weights, rtol=1e-12, atol=1e-14)
replay = {**reloaded, 'estimators': []}
replay_X = reloaded['scaler'].transform(saved[FEATURES].to_numpy())
with threadpool_limits(limits=1):
    for j, reason in enumerate(REASONS):
        target = saved[reason].to_numpy(dtype=int)
        weight = saved[f'normalized_weight_{reason}'].to_numpy()
        included = weight > 0
        estimator = clone(reloaded['estimators'][j])
        with warnings.catch_warnings():
            warnings.simplefilter('error', ConvergenceWarning)
            estimator.fit(replay_X[included], target[included], sample_weight=weight[included])
        replay['estimators'].append(estimator)
np.testing.assert_allclose(predict_model(replay, XT), test_probability, rtol=1e-8, atol=1e-10)
for relative, digest in PROTECTED.items():
    assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest() == digest
print('Verified submission schema/order/labels, saved model reload, independent refit, and unchanged source artifacts.')



Verified submission schema/order/labels, saved model reload, independent refit, and unchanged source artifacts.


## Save final fitting provenance

Record the recipe, generated-label counts, file hashes, and manual upload status. There is no remaining independent local validation score for this refit.

In [7]:
submission_sha = hashlib.sha256(submission_path.read_bytes()).hexdigest()
manifest_path = OUT/'submission_manifest.json'
previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
same_file = previous.get('submission_sha256') == submission_sha
manifest = dict(experiment='final_dbscan_trainval235', model_file=str(model_path.relative_to(DATA)),
                model_sha256=hashlib.sha256(model_path.read_bytes()).hexdigest(),
                submission_file=str(submission_path.relative_to(DATA)), submission_sha256=submission_sha,
                training_verified_rows=235, original_training_verified_rows=157, former_validation_rows=78,
                originally_unlabeled_rows=1014, clustering_rows=1249,
                generated_trials=int(((raw_weights > 0).any(axis=1) & ~known).sum()),
                generated_reason_slots=int(((raw_weights > 0) & ~known[:, None]).sum()),
                test_rows=312, test_fallback_trials=int(empty.sum()),
                changed_test_trials_vs_157_label_model=changed_test_trials,
                recipe=RECIPE, fitted_dbscan_eps=eps,
                historical_selection_validation=dict(verified_training_rows=157, validation_rows=78,
                                                     sample_f1=historical_sample, macro_f1=historical_macro),
                final_model_independent_validation_f1=None,
                kaggle_score=previous.get('kaggle_score') if same_file else None,
                kaggle_status=previous.get('kaggle_status', 'ready for manual upload; not submitted') if same_file
                              else 'ready for manual upload; not submitted',
                upload_performed_by_agent=False, final_refit_completed_utc=datetime.now(timezone.utc).isoformat(),
                protected_input_sha256=PROTECTED,
                dataset_file_sha256={str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest()
                                     for p in sorted((OUT/'datasets').glob('*.csv'))},
                versions=dict(python=platform.python_version(), sklearn=sklearn.__version__,
                              numpy=np.__version__, pandas=pd.__version__))
manifest_path.write_text(json.dumps(manifest, indent=2)+'\n')
lines = ['# Final DBSCAN fit: 235 verified labels', '',
         f'Final-fit submission CSV: `part1/{manifest["submission_file"]}`. '
         f'Kaggle score: {manifest["kaggle_score"]}; status: {manifest["kaggle_status"]}. No upload was performed by the agent.', '',
         '## Frozen recipe and final training', '',
         'The current validation-selected DBSCAN recipe is refitted using all 235 verified labels '
         '(157 original training + 78 former validation). The original 1,014 unlabeled training trials remain available. '
         'Test trials never enter preprocessing, clustering, pseudo-label generation, or classifier fitting.', '',
         'Classifier StandardScaler is fitted on 235 verified rows. Cluster StandardScaler and DBSCAN '
         'are fitted on the combined 1,249 training/validation rows. DBSCAN retains min_samples=5 and '
         'the training-derived q25 fifth-neighbor-radius rule (floor 1e-6); its numerical radius is '
         f'relearned as {eps:.15g}. Labels/weights are regenerated from the 235 known labels using unchanged '
         'support, confidence, smoothing, cap, and class-budget rules. Five L2 logistic classifiers retain '
         'C=1000, solver=lbfgs, tol=1e-4, max_iter=5000, class_weight=None, random_state=42. '
         'Fitting weights sum to 235 per reason. Threshold 0.5 and highest-probability fallback stay fixed.', '',
         '## Generated labels', '',
         '| Reason | Verified positives | Verified negatives | Generated positives | Generated negatives | Fitting rows |',
         '|---|---:|---:|---:|---:|---:|']
for row in counts.itertuples():
    lines.append(f'| {row.reason} | {row.verified_positive} | {row.verified_negative} | '
                 f'{row.generated_positive} | {row.generated_negative} | {row.total_fitting_rows} |')
lines += ['', '## Submission checks and evaluation provenance', '',
          f'312 unique test IDs in sample-submission order; columns ID/Predicted; valid nonempty label strings. '
          f'Fallback on {int(empty.sum())} trials. Predictions change on {changed_test_trials} test trials '
          'relative to the earlier DBSCAN model fitted with 157 verified labels. Saved models reload and '
          'the five classifiers independently refit from the persisted data/weights with matching probabilities. '
          'All protected original model, data, and submission checksums remain unchanged.', '',
          f'The historical pre-refit validation sample F1 is {historical_sample:.6f}, macro F1 '
          f'{historical_macro:.6f}. These scores describe the earlier 157-label model. The new final model '
          'has trained on the former validation labels and has no independent local validation score; '
          f'its recorded Kaggle score is {manifest["kaggle_score"]}. No new settings were chosen during this refit.', '',
          f'Submission SHA-256: `{submission_sha}`.', '',
          'Reproduce using part1/final_dbscan_trainval235.ipynb. datasets/ preserves exact raw embeddings, '
          'combined verified labels, generated targets, raw/normalized weights, test features, and source audits. '
          'models/ stores the final labeler and classifiers. predictions/ stores test probabilities, labels, '
          'and fallback flags. submission_manifest.json records provenance, recipe, checksums, and manual upload status.']
(OUT/'SUMMARY.md').write_text('\n'.join(lines)+'\n')
print('Final-fit submission artifact:', submission_path)
print('SHA-256:', submission_sha)


Final submission ready: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/submissions/ready/part1_dbscan_logistic_trainval235.csv
SHA-256: 19f29c7bcbda974f8ae9646d3ced83bad2953ead45412e88731e8c12e55d2661
